# 01 · Data pipeline

Owner: data lead.

Runs stage 1 (clean + filter) and stage 2 (pairing) on the raw files, then the quality checks from the proposal.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, build_dataset, models, evaluate

## Stage 1 — clean, calendar-check, filter, validate (one Parquet per raw file)

In [ ]:
# Stage 1 + 2 from the project config (same as `python -m src.build_dataset --config configs/main_2016_2020.yaml`)
cfg = build_dataset.config_from_dict(build_dataset.load_config(ROOT / "configs" / "main_2016_2020.yaml"))
cal = build_dataset.trading_days()
report = build_dataset.run_stage1(cfg, cal)
report.tail()

In [ ]:
from IPython.display import Markdown
Markdown((PROCESSED_DIR / "data_quality.md").read_text())

## Stage 2 — pair on the NYSE calendar, build target, assign splits

In [ ]:
summary = build_dataset.run_stage2(cfg, cal)
summary

## Quality checks

- filtered contracts per day (a dip flags a bad file; expect one in March 2020)
- Black-Scholes one-day hedging error distribution
- vendor delta vs. delta recomputed from the vendor's IV

In [ ]:
pairs = pd.read_parquet(PAIRS_PATH)
pairs.groupby("date").size().plot(title="pairs per start day"); plt.show()
pairs.groupby("year")["err_bs"].agg(["count", "mean", "std"])

In [ ]:
from src.bs import bs_delta
chk = pairs.sample(20000, random_state=0)
d_re = bs_delta(chk.S, chk.strike, chk["T"], 0.01, chk.iv, chk["type"].to_numpy())
plt.scatter(chk.delta, d_re, s=2, alpha=.3); plt.plot([-1,1],[-1,1],'r'); plt.xlabel("vendor delta"); plt.ylabel("recomputed"); plt.show()